# Optional: Test and Promote a Newly Retrained Face Recognition Model

> **Standard demo guardrail:** skip this notebook when the demo should use the last
> validated model version. It can upload a new ONNX artifact to MinIO and restart the
> model server if promotion is explicitly enabled.

In this notebook we test the ONNX model exported in notebook 02 on **still images** and **video**.

The retrained model has learned to distinguish *your* face from other faces:
- **Green** bounding boxes → your face (the class the model was trained on)
- **Red** bounding boxes → other / unknown faces

We'll run inference locally using the Ultralytics YOLO runtime with the ONNX backend.
Promotion to the model server is disabled by default so the validated demo deployment is not overwritten.


In [ ]:
!pip install --no-cache-dir -q -r requirements.txt

from ultralytics import YOLO
import numpy as np
import cv2
from matplotlib import pyplot as plt
from pathlib import Path
from IPython.display import Video, display, Image as IPImage
import remote_infer

## Load the retrained ONNX model

This section is only for a model produced by notebook 02. The standard demo does not need a local ONNX file; it queries the served validated model in notebook 04.


In [ ]:
import hashlib
import os
from botocore.config import Config
import boto3

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Force ONNX to use CPU for inference

def find_latest_local_onnx():
    runs_dir = Path("runs")
    if not runs_dir.exists():
        return None
    candidates = [
        path for path in runs_dir.rglob("weights/best.onnx")
        if "face-recognition" in path.parts
    ]
    return max(candidates, key=lambda path: path.stat().st_mtime) if candidates else None

def download_served_model():
    bucket = os.getenv("FACE_RECOGNITION_BUCKET", "models")
    key = os.getenv("FACE_RECOGNITION_MODEL_KEY", "face-recognition/1/model.onnx")
    endpoint = os.getenv("MINIO_ENDPOINT", "http://minio.minio-storage.svc:9000")
    destination = Path("/tmp/face-recognition-served-model.onnx")
    s3 = boto3.client(
        "s3",
        endpoint_url=endpoint,
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "rhoai-access-key"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "rhoai-secret-key-12345"),
        config=Config(signature_version="s3v4"),
    )
    s3.download_file(bucket, key, str(destination))
    return destination, f"s3://{bucket}/{key}"

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

onnx_path = find_latest_local_onnx()
if onnx_path is None:
    print("Local trained ONNX not found. Loading the currently served validated model from MinIO...")
    print("Run notebook 02 first if you want notebook 03 to test a newly trained local model.")
    onnx_path, model_source = download_served_model()
else:
    model_source = "local notebook 02 training output"

model = YOLO(str(onnx_path), task="detect")
print(f"Model source: {model_source}")
print(f"Model loaded from: {onnx_path}")
print(f"SHA256: {sha256_file(onnx_path)}")
print(f"Class names: {model.names}")

In [ ]:
# ============================================================
# INFERENCE CONFIGURATION — tweak these to tune results
# ============================================================
CONF_THRESHOLD = 0.7   # 0.5=sensitive, 0.6=sensitive, 0.7=balanced (default)
IOU_THRESHOLD  = 0.45  # NMS overlap — lower=fewer overlapping boxes
IDENTITY_DEDUP = True  # Apply one-adnan-per-frame constraint

print(f"Inference config: conf={CONF_THRESHOLD}, iou={IOU_THRESHOLD}, dedup={IDENTITY_DEDUP}")


In [ ]:
def deduplicate_adnan(results, adnan_class_id=0):
    """Keep only the highest-confidence adnan detection per image.
    Any additional adnan detections are reclassified as unknown_face.
    Rationale: one person cannot appear twice in the same photo."""
    for result in results:
        boxes = result.boxes
        if boxes is None or len(boxes) == 0:
            continue
        cls = boxes.cls.cpu().numpy()
        conf = boxes.conf.cpu().numpy()
        adnan_indices = [i for i, c in enumerate(cls) if int(c) == adnan_class_id]
        if len(adnan_indices) > 1:
            best = max(adnan_indices, key=lambda i: conf[i])
            # Modify the raw data tensor (x1,y1,x2,y2,conf,cls)
            new_data = boxes.data.clone()
            for idx in adnan_indices:
                if idx != best:
                    new_data[idx, 5] = 1  # reclassify as unknown_face
            result.boxes = type(boxes)(new_data, result.orig_shape)
            n = len(adnan_indices) - 1
            print(f"    Dedup: kept 1 adnan ({conf[best]:.2f}), reclassified {n} as unknown")
    return results

print("Post-processing: deduplicate_adnan() loaded")
print("  Only the highest-confidence adnan detection per image is kept.")
print("  Additional adnan detections are reclassified as unknown_face.")


## Test on still images

In [ ]:
import glob

face_images = sorted(glob.glob("images/test_face_*.jpg"))
print(f"Testing on {len(face_images)} face images\n")

for image_path in face_images:
    results = model.predict(image_path, conf=CONF_THRESHOLD)
    if IDENTITY_DEDUP:
        results = deduplicate_adnan(results)
    result = results[0]

    print(f"--- {image_path} ---")
    for box in result.boxes:
        class_id = int(box.cls.item())
        class_name = result.names[class_id]
        conf = round(box.conf.item(), 2)
        print(f"  Detected: {class_name} ({conf})")

    img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    fig = plt.figure(figsize=(8, 5))
    plt.axis("off")
    plt.imshow(img)
    plt.title(Path(image_path).name)
    plt.show()

In [ ]:
group_images = sorted(glob.glob("images/test_group_*.jpg"))
print(f"Testing on {len(group_images)} group images\n")

for image_path in group_images:
    results = model.predict(image_path, conf=CONF_THRESHOLD)
    if IDENTITY_DEDUP:
        results = deduplicate_adnan(results)
    result = results[0]

    print(f"--- {image_path} ---")
    for box in result.boxes:
        class_id = int(box.cls.item())
        class_name = result.names[class_id]
        conf = round(box.conf.item(), 2)
        print(f"  Detected: {class_name} ({conf})")

    img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
    fig = plt.figure(figsize=(10, 6))
    plt.axis("off")
    plt.imshow(img)
    plt.title(Path(image_path).name)
    plt.show()

## Video Inference — The Demo Highlight

Now for the fun part: we process a video where the model identifies **your face** versus other faces in real-time.

Each frame is run through the ONNX model locally. Your face should appear with a **green** bounding box,
while other faces get a **red** box. The annotated frames are reassembled into an output video.

In [ ]:
video_path = "videos/test_group_video.mov"

if Path(video_path).exists():
    print("Processing video — this may take a minute...")
    output_path = remote_infer.process_video_local(video_path, model, conf=CONF_THRESHOLD)
    # Embed video as base64 for reliable JupyterLab playback
    import base64
    from IPython.display import HTML
    with open(output_path, "rb") as vf:
        video_b64 = base64.b64encode(vf.read()).decode()
    display(HTML(f'<video controls style="max-width:480px; border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.3)"><source src="data:video/mp4;base64,{video_b64}" type="video/webm"></video>'))
else:
    print(f"Video not found: {video_path}")
    print("Place a test video (10-30s, you + another person) in the videos/ folder.")

In [ ]:
video_path = "videos/test_face_video.mov"

if Path(video_path).exists():
    print("Processing solo face video...")
    output_path = remote_infer.process_video_local(video_path, model, conf=CONF_THRESHOLD)
    import base64
    from IPython.display import HTML
    with open(output_path, "rb") as vf:
        video_b64 = base64.b64encode(vf.read()).decode()
    display(HTML(f'<video controls style="max-width:480px; border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.3)"><source src="data:video/mp4;base64,{video_b64}" type="video/webm"></video>'))
else:
    print(f"Video not found: {video_path}")


In [ ]:
video_path = "videos/test_group_video_02.mov"

if Path(video_path).exists():
    print("Processing group video 02...")
    output_path = remote_infer.process_video_local(video_path, model, conf=CONF_THRESHOLD)
    import base64
    from IPython.display import HTML
    with open(output_path, "rb") as vf:
        video_b64 = base64.b64encode(vf.read()).decode()
    display(HTML(f'<video controls style="max-width:480px; border-radius:8px; box-shadow:0 2px 8px rgba(0,0,0,0.3)"><source src="data:video/mp4;base64,{video_b64}" type="video/webm"></video>'))
else:
    print(f"Video not found: {video_path}")


## Optional: promote a newly trained model to the model server

Promotion is disabled by default. Leave it disabled for the standard demo so the
`face-recognition` InferenceService keeps serving the last validated model version.

Only set `PROMOTE_TO_MODEL_SERVER = True` when you intentionally want to replace
`s3://models/face-recognition/1/model.onnx` with the local `best.onnx` produced by notebook 02.


In [ ]:
import boto3
from pathlib import Path
from botocore.config import Config

PROMOTE_TO_MODEL_SERVER = False

if not PROMOTE_TO_MODEL_SERVER:
    print("Promotion skipped. The standard demo uses the last validated model already served by RHOAI.")
    print("Set PROMOTE_TO_MODEL_SERVER = True only when intentionally replacing the served model artifact.")
else:
    if "find_latest_local_onnx" in globals():
        onnx_path = find_latest_local_onnx()
    else:
        onnx_candidates = [
            path for path in Path("runs").rglob("weights/best.onnx")
            if "face-recognition" in path.parts
        ]
        onnx_path = max(onnx_candidates, key=lambda path: path.stat().st_mtime) if onnx_candidates else None
    if onnx_path is None:
        raise FileNotFoundError("No trained ONNX found. Run notebook 02 first.")

    BUCKET = "models"
    KEY = "face-recognition/1/model.onnx"

    s3 = boto3.client("s3",
        endpoint_url="http://minio.minio-storage.svc:9000",
        aws_access_key_id="rhoai-access-key",
        aws_secret_access_key="rhoai-secret-key-12345",
        config=Config(signature_version="s3v4"))

    print(f"Uploading {onnx_path} -> s3://{BUCKET}/{KEY}...")
    s3.upload_file(str(onnx_path), BUCKET, KEY)

    obj = s3.head_object(Bucket=BUCKET, Key=KEY)
    size_mb = obj["ContentLength"] / (1024 * 1024)
    print(f"Uploaded: s3://{BUCKET}/{KEY} ({size_mb:.1f} MB)")


In [ ]:
import subprocess, time

if not globals().get("PROMOTE_TO_MODEL_SERVER", False):
    print("Model server restart skipped because PROMOTE_TO_MODEL_SERVER is False.")
else:
    print("Restarting model server to load the new model...")
    result = subprocess.run(
        ["oc", "delete", "pod", "-n", "enterprise-mlops",
         "-l", "serving.kserve.io/inferenceservice=face-recognition"],
        capture_output=True, text=True
    )
    print(result.stdout.strip() or result.stderr.strip())

    print("Waiting for predictor pod to become Ready...")
    for i in range(24):
        time.sleep(5)
        check = subprocess.run(
            ["oc", "get", "inferenceservice", "face-recognition", "-n", "enterprise-mlops",
             "-o", 'jsonpath={.status.conditions[?(@.type=="Ready")].status}'],
            capture_output=True, text=True
        )
        status = check.stdout.strip()
        if status == "True":
            print(f"Model server Ready (took ~{(i+1)*5}s)")
            break
        if i % 3 == 0:
            print(f"  Waiting... ({(i+1)*5}s)")
    else:
        print("WARNING: Model server not Ready after 120s. Check: oc get inferenceservice face-recognition -n enterprise-mlops")


## Done!

Local testing is complete.

For the standard demo, the served model was not modified. Open **`04-query-model-server.ipynb`** to query the last validated model version via the production REST API.

If you intentionally enabled promotion, verify the new model server state before continuing.
